In [14]:
api_url = "https://earth-search.aws.element84.com/v1"

from pystac_client import Client # pystac_client library ka use STAC API se connect hone aur data search karne ke liye hota hai.

client = Client.open(api_url) # client is now connected to the STAC API at the api_url. It can be used to search for and retrieve geospatial data.

collections = client.get_collections() #satellite datasets ka group.

for i in collections:
    print(i.id)

sentinel-2-pre-c1-l2a
cop-dem-glo-30
naip
cop-dem-glo-90
landsat-c2-l2
sentinel-2-l2a
sentinel-2-l1c
sentinel-2-c1-l2a
sentinel-1-grd


In [15]:
collection_sentinel_2_l2a = "sentinel-2-l2a"

In [16]:
from shapely.geometry import Point 

point = Point(27.95, 36.20)  # Coordinates of a point on Rhodes(for our case study); Point(longitude, latitude)

# All the data we are dealing with right now is metadata, but even metadata can be quite bulky if a large number of scenes match our search. For this reason, we limit the search by the intersection of the point (by setting the parameter intersects) and assign the collection (by setting the parameter collections)

In [28]:
search = client.search(collections = [
                        collection_sentinel_2_l2a], # sirf Sentinel-2 dataset me search karo.
                        intersects = point, # jis satellite image ka area is point ko cover karta ho, usko return karo.
                        datetime = '2023-07-01/2023-08-31',  # 1st of July 2023 and 31st of August 2023
                        query = ["eo:cloud_cover<1"]
                        )  

print(search.matched()) 

11


In [ ]:
items = search.item_collection()
items.save_object('rhodes_sentinel-2.json') #file is created in GeoJSON format. this file contains the metadata of the files that meet out criteria
print(len(items))

for i in items:
    print(i.id) #Each of the items contains information about the scene geometry

11
S2A_35SNA_20230827_0_L2A
S2B_35SNA_20230822_0_L2A
S2A_35SNA_20230817_0_L2A
S2B_35SNA_20230812_0_L2A
S2A_35SNA_20230807_0_L2A
S2B_35SNA_20230802_0_L2A
S2A_35SNA_20230728_0_L2A
S2B_35SNA_20230723_0_L2A
S2A_35SNA_20230718_0_L2A
S2B_35SNA_20230713_0_L2A
S2A_35SNA_20230708_0_L2A


In [30]:
item = items[0]
print(item.datetime)
print(item.geometry)
print(item.properties)

2023-08-27 09:00:21.327000+00:00
{'type': 'Polygon', 'coordinates': [[[27.290401625602243, 37.04621863329741], [27.23303872472207, 36.83882218126937], [27.011145718480538, 36.05673246264742], [28.21878905911668, 36.05053734221328], [28.234426643135546, 37.04015200857309], [27.290401625602243, 37.04621863329741]]]}
{'created': '2023-08-27T18:15:43.106Z', 'platform': 'sentinel-2a', 'constellation': 'sentinel-2', 'instruments': ['msi'], 'eo:cloud_cover': 0.955362, 'proj:epsg': 32635, 'mgrs:utm_zone': 35, 'mgrs:latitude_band': 'S', 'mgrs:grid_square': 'NA', 'grid:code': 'MGRS-35SNA', 'view:sun_azimuth': 144.36354987218, 'view:sun_elevation': 59.06665363921, 's2:degraded_msi_data_percentage': 0.0126, 's2:nodata_pixel_percentage': 12.146327, 's2:saturated_defective_pixel_percentage': 0, 's2:dark_features_percentage': 0.249403, 's2:cloud_shadow_percentage': 0.237454, 's2:vegetation_percentage': 6.073786, 's2:not_vegetated_percentage': 18.026696, 's2:water_percentage': 74.259061, 's2:unclassif

In [33]:
import pystac
items_loaded = pystac.ItemCollection.from_file("rhodes_sentinel-2.json") #to load the file we created in the previous step, we use the from_file method of the ItemCollection class. This method takes the path to the file as an argument and returns an ItemCollection object that contains all the items in the file.

So far we have only worked with metadata - but how can one get to the actual images of a satellite scene (the “assets” in the STAC nomenclature)? These can be reached via links that are made available through the item’s attribute assets

In [34]:
assests = items[-1].assets